In [1]:

import json
import os
import pandas as pd
from pathlib import Path
from openai import OpenAI


In [2]:


ROOT      = Path("../")
JSON_PATH = ROOT / "data" / "yt_data" / "dance_youtube_data.json"
CSV_PATH  = ROOT / "data" / "yt_data" / "dance_videos.csv"
KEY_PATH  = ROOT / "data" / "GPT4oKEY.txt"

api_key = KEY_PATH.read_text(encoding="utf-8").strip()
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)
MODEL = "openai/gpt-4o-mini"
print(f"Client ready → {MODEL}")

Client ready → openai/gpt-4o-mini


In [3]:

with JSON_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

# Find video_ids that already have sentiment scores in dance_videos.csv
existing_csv = pd.read_csv(CSV_PATH)
already_scored = set(
    existing_csv.dropna(subset=["comment_sentiment"])["video_id"].astype(str).unique()
)
print(f"Videos already scored: {len(already_scored)} — skipping these.")

# Flatten into a list of {video_id, text} dicts — only for unscored videos
comment_rows = []
for entry in data.values():
    for video in entry.get("videos", []):
        video_id = (
            video.get("basic_info", {}).get("video_id")
            or video.get("comprehensive_info", {}).get("video_id")
        )
        if not video_id or video_id in already_scored:
            continue
        for comment in video.get("comments", []):
            text = str(comment.get("text", "")).strip()
            if text:
                comment_rows.append({"video_id": video_id, "text": text})

comments_df = pd.DataFrame(comment_rows)
print(f"Total comments to score: {len(comments_df)}")
print(f"Unique videos to score:  {comments_df['video_id'].nunique()}")
comments_df.head(3)

Videos already scored: 403 — skipping these.
Total comments to score: 966
Unique videos to score:  100


,video_id,text
0,lrluSO-Qs0E,Learning this bcs of sung hanbin btw AHAHAHA
1,lrluSO-Qs0E,Naughty by Red Velvet Irene & Seulgi got me wa...
2,lrluSO-Qs0E,"I was fine until he said ""now let's go a littl..."


In [4]:

# sentiment using openrouter

def score_comment(text: str) -> tuple[str, float]:
    """Returns (label, score) for a single comment. label: POSITIVE/NEUTRAL/NEGATIVE, score: 1/0/-1."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Classify the sentiment of a YouTube comment about a dance video. "
                    "Reply with exactly one word: POSITIVE, NEUTRAL, or NEGATIVE."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    label = response.choices[0].message.content.strip().upper()
    if label not in ("POSITIVE", "NEUTRAL", "NEGATIVE"):
        label = "NEUTRAL"
    score_map = {"POSITIVE": 1.0, "NEUTRAL": 0.0, "NEGATIVE": -1.0}
    return label, score_map[label]

print("Sentiment function ready.")

Sentiment function ready.


In [5]:

sentiments = []
texts = comments_df["text"].tolist()

for idx, text in enumerate(texts):
    label, score = score_comment(text)
    sentiments.append({"label": label, "score": score})

    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} / {len(texts)} comments...")

print(f"\nDone — {len(texts)} comments processed.")

comments_df["sentiment_label"] = [s["label"] for s in sentiments]
comments_df["score"]           = [s["score"] for s in sentiments]
comments_df.head(5)

Processed 10 / 966 comments...
Processed 20 / 966 comments...
Processed 30 / 966 comments...
Processed 40 / 966 comments...
Processed 50 / 966 comments...
Processed 60 / 966 comments...
Processed 70 / 966 comments...
Processed 80 / 966 comments...
Processed 90 / 966 comments...
Processed 100 / 966 comments...
Processed 110 / 966 comments...
Processed 120 / 966 comments...
Processed 130 / 966 comments...
Processed 140 / 966 comments...
Processed 150 / 966 comments...
Processed 160 / 966 comments...
Processed 170 / 966 comments...
Processed 180 / 966 comments...
Processed 190 / 966 comments...
Processed 200 / 966 comments...
Processed 210 / 966 comments...
Processed 220 / 966 comments...
Processed 230 / 966 comments...
Processed 240 / 966 comments...
Processed 250 / 966 comments...
Processed 260 / 966 comments...
Processed 270 / 966 comments...
Processed 280 / 966 comments...
Processed 290 / 966 comments...
Processed 300 / 966 comments...
Processed 310 / 966 comments...
Processed 320 / 9

,video_id,text,sentiment_label,score
0,lrluSO-Qs0E,Learning this bcs of sung hanbin btw AHAHAHA,POSITIVE,1.0
1,lrluSO-Qs0E,Naughty by Red Velvet Irene & Seulgi got me wa...,POSITIVE,1.0
2,lrluSO-Qs0E,"I was fine until he said ""now let's go a littl...",NEUTRAL,0.0
3,lrluSO-Qs0E,naughty: the kpop stans will understand,NEUTRAL,0.0
4,lrluSO-Qs0E,finally one step I can practice by staying in bed,NEUTRAL,0.0


In [6]:

def majority_label(labels):
    return labels.value_counts().idxmax().lower()

agg = (
    comments_df
    .groupby("video_id")
    .agg(
        comment_sentiment               = ("sentiment_label", majority_label),
        comment_sentiment_avg_compound  = ("score", lambda x: round(x.mean(), 4)),
        comment_sentiment_comment_count = ("text", "count"),
    )
    .reset_index()
)

print(f"Aggregated sentiment for {len(agg)} videos.")
agg.head()

Aggregated sentiment for 100 videos.


,video_id,comment_sentiment,comment_sentiment_avg_compound,comment_sentiment_comment_count
0,-FvsnqL124Q,positive,0.9,10
1,-IdY7BQTCWs,positive,0.6,10
2,-JCJUaoRPLI,positive,0.9,10
3,-jM6bHqPBEk,positive,0.7,10
4,0DPHwlgqaMY,positive,1.0,1


In [7]:

# ── Merge into dance_videos.csv ────────────────────────────────────────────────
videos_df = pd.read_csv(CSV_PATH)
print(f"CSV rows before merge: {len(videos_df)}")

if agg.empty:
    print("No new sentiment scores to add — CSV unchanged.")
else:
    # Only merge sentiment for videos that don't have it yet
    sentiment_cols = [
        "comment_sentiment",
        "comment_sentiment_avg_compound",
        "comment_sentiment_comment_count",
    ]
    merged = videos_df.merge(agg, on="video_id", how="left", suffixes=("", "_new"))

    for col in sentiment_cols:
        new_col = col + "_new"
        if new_col in merged.columns:
            # Fill only where the original value is missing
            merged[col] = merged[col].combine_first(merged[new_col])
            merged.drop(columns=[new_col], inplace=True)

    merged.to_csv(CSV_PATH, index=False, encoding="utf-8")
    print(f"CSV saved  → {CSV_PATH}")
    print(f"Total rows : {len(merged)} | New videos scored: {agg['video_id'].nunique()}")
    merged[["video_id", "comment_sentiment", "comment_sentiment_avg_compound",
            "comment_sentiment_comment_count"]].dropna().head(10)


CSV rows before merge: 606
CSV saved  → ../data/yt_data/dance_videos.csv
Total rows : 606 | New videos scored: 100
